In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MINIO_ACCESS = "slavakoder"
MINIO_SECRET = "slavakoder"
DB_PASS = "airflow"

spark = SparkSession.builder \
    .appName('cleandata') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.shuffle.partitions', '8') \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "jdbc") \
    .config("spark.sql.catalog.demo.uri", "jdbc:postgresql://postgres:5432/airflow") \
    .config("spark.sql.catalog.demo.jdbc.user", "airflow") \
    .config("spark.sql.catalog.demo.jdbc.password", DB_PASS) \
    .config("spark.sql.catalog.demo.warehouse", "s3a://raw-bronze/warehouse") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.6.0") \
    .getOrCreate()
print('запускаемся')

spark.sparkContext.setLogLevel('WARN')

raw_data = "s3a://raw-bronze/landing/p2p_transfers/chunk_1.csv"

ddl_schema = "tx_id STRING, sender_id STRING, receiver_id STRING, amount STRING, currency STRING, status STRING, timestamp LONG"

df = spark.read.csv(raw_data, header=True, schema=ddl_schema)

запускаемся


In [4]:
df = (df
    .withColumn('timestamp', F.from_unixtime(F.col('timestamp')).cast('timestamp'))
    .withColumn('sender_id', F.regexp_replace(F.col('sender_id'), r'\s+', ''))
    .withColumn('receiver_id', F.regexp_replace(F.col('receiver_id'), r'\s+', ''))
    .withColumn('amount', F.regexp_replace(F.col('amount'), ',', '.').cast('double'))
    .withColumn('timestamp', F.coalesce(F.col('timestamp'), F.lit('1970-01-01 00:00:00')))
    .withColumn('status', F.coalesce(F.col('status'), F.lit('UNKNOWN')))
)
df = df.fillna('1970-01-01 00:00:00', subset=['timestamp'])
df = df.dropDuplicates(['tx_id'])
df = df.replace(['', 'N/A', 'NULL ', 'NULL'], 'Unknown', subset=['status'])

In [5]:
df.show(20, truncate=False)

+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|tx_id                           |sender_id|receiver_id|amount |currency|status  |timestamp          |
+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|00018318f9c7404b88e9b1a8a42e87de|USR_26357|USR_13484  |1609.97|USDT    |Unknown |1970-01-01 00:00:00|
|0001f69016be4938bb90ec7bdcc3c021|USR_32029|USR_39636  |1177.49|USDT    |PENDING |1970-01-01 00:00:00|
|000235c868b84dd281531a43ff8b5b6f|USR_12566|USR_20714  |2374.66|RUB     |SUCCESS |1970-01-01 00:00:00|
|0002451d01424129ab077efd753161ea|USR_39981|USR_20259  |451.7  |USD     |SUCCESS |1970-01-01 00:00:00|
|0002527e5b564e498e890c5e6cd7b72a|USR_5809 |USR_47442  |4892.09|EUR     |PENDING |1970-01-01 00:00:00|
|00028bc1860c4bb39c3076ab0f50e92a|USR_38867|USR_37548  |541.11 |KZT     |SUCCESS |1970-01-01 00:00:00|
|0002c3918e144733b3c7593c8aafb418|USR_28720|USR_35799  |3470.06|USD     |

In [ ]:
df.printSchema()